In [1]:
import asyncio
import logging
import os
from pathlib import Path
import yaml
import pandas as pd
from typing import Dict, Any, Tuple, List, Optional
from dotenv import load_dotenv
import argparse
from datetime import datetime

from src.factories.clothing_factory import ClothingFactory
from src.services.api_service import APIService
from src.services.attribute_generator import AttributeGenerator
from src.services.batch_processor import BatchProcessor
from src.utils.logging_utils import (
    setup_logging, 
    LoggingContext, 
    PerformanceMetrics,
    StructuredLogger
)
from src.services.product_standardizer import ProductStandardizer
from src.base.clothing_item import ClothingItem


In [2]:
from abc import ABC, abstractmethod
from enum import Enum
from typing import List, Dict, Any, Union
import re
import logging
from pydantic import (
    BaseModel, Field, field_validator, ValidationInfo
)
from src.base.enums import (
    Color, ColorDetailed, Pattern, Material, 
    EmbellishmentLevel, Embellishment, Occasion, 
    Style, Gender, AgeGroup
)
from src.utils.validation import validate_enum_field, validate_enum_list
from src.utils.config_loader import ConfigManager
import pandas as pd


In [9]:
class ConfigLoader:
    def __init__(self, config_dir: Path):
        self.config_dir = config_dir
        
    def load(self) -> Dict[str, Any]:
        """Load base configuration from YAML file"""
        config_path = self.config_dir / "base_config.yaml"
        with open(config_path) as f:
            return yaml.safe_load(f)


def load_config(config_path: str) -> Dict:
    with open(config_path) as f:
        return yaml.safe_load(f)

In [10]:
base_path = '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/product_attributes/config/base_config.yaml'
config = load_config(base_path)
from src.utils.config_loader import ConfigManager

FileNotFoundError: [Errno 2] No such file or directory: '/Users/aswinsreenivas/Projects/D2C Search Engine/uplyft_server/product_attributes/config/base_config.yaml'

## Load products

In [8]:
#!/usr/bin/env python3

import json
import os
from typing import Dict, Any, Optional


def float_hook(dct: dict) -> dict:
    for k, v in dct.items():
        if isinstance(v, str):
            try:
                if 'e' in v.lower():
                    dct[k] = float(v)
            except ValueError:
                pass
        elif isinstance(v, list):
            for i, item in enumerate(v):
                if isinstance(item, str) and 'e' in item.lower():
                    try:
                        v[i] = float(item)
                    except ValueError:
                        pass
    return dct


def count_unique_products(file_path: str) -> int:
    try:
        all_data = {}
        current_obj = ""
        brace_count = 0
        total_objects_found = 0
        failed_objects = 0
        
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                brace_count += line.count('{') - line.count('}')
                current_obj += line
                
                if brace_count == 0 and current_obj:
                    total_objects_found += 1
                    try:
                        data = json.loads(current_obj, object_hook=float_hook)
                        if data:
                            product_id = next(iter(data))
                            all_data[product_id] = data[product_id]
                    except json.JSONDecodeError:
                        failed_objects += 1
                        preview_len = 75
                        truncated = len(current_obj) > preview_len
                        preview = current_obj[:preview_len]
                        if truncated:
                            preview += "..."
                        obj_num = total_objects_found
                        print(f"\nFailed to parse object {obj_num}:")
                        print(preview)
                    current_obj = ""
        
        product_count = len(all_data)
        print("\nParsing Statistics:")
        print(f"Total JSON objects found: {total_objects_found}")
        print(f"Failed to parse: {failed_objects}")
        print(f"Unique products found: {product_count}")
        
        if product_count > 0:
            print("\nFirst few product IDs found:")
            for pid in list(all_data.keys())[:5]:
                print(f"- {pid}")
                
        return product_count
            
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return 0
    except Exception as e:
        print(f"Error: {str(e)}")
        return 0


def load_product_by_id(file_path: str, product_id: str) -> Optional[Dict[str, Any]]:
    try:
        current_obj = ""
        brace_count = 0
        
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                brace_count += line.count('{') - line.count('}')
                current_obj += line
                
                if brace_count == 0 and current_obj:
                    try:
                        data = json.loads(current_obj, object_hook=float_hook)
                        if data and product_id in data:
                            return data[product_id]
                    except json.JSONDecodeError:
                        pass
                    current_obj = ""
        
        print(f"Product ID {product_id} not found")
        return None
            
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"Error: {str(e)}")
        return None


json_path = 'D:/OneDrive - StarHub Ltd/5. Scripts/product-attributes/output/Saree_attributes.json'
count = count_unique_products(json_path)
print(f"\nFinal count of unique products: {count}")


Parsing Statistics:
Total JSON objects found: 25
Failed to parse: 0
Unique products found: 25

First few product IDs found:
- 7372922781761
- 7372876546113
- 7372801835073
- 7372794265665
- 7372774015041

Final count of unique products: 25


In [9]:
product_temp = load_product_by_id(json_path, '7356361736257')
if product_temp:
    product_temp.pop('text_embedding', None)
    product_temp.pop('image_embedding', None)
product_temp

{'primary_color': 'Red',
 'primary_color_detailed': 'Crimson',
 'secondary_colors': [],
 'secondary_colors_detailed': [],
 'color_pairings_hex': ['#FF0033', '#FFD700', '#FFC0CB', '#800000', '#FF69B4'],
 'pattern': [],
 'saree_type': 'Banarasi',
 'border_width': 'Wide',
 'border_design': 'Zari',
 'border_design_details': ['Intricate zari work on border',
  'Traditional motifs in gold',
  'Scalloped edge design',
  'Contrast red and gold pattern',
  'Rich metallic sheen'],
 'pallu_design': 'Elaborate',
 'pre_draped': False,
 'material': 'Silk',
 'length': 6.3,
 'width': 1.1,
 'weight': 800,
 'care_instructions': 'Dry clean only. Store in a cool, dry place. Avoid direct sunlight.',
 'occasions': [],
 'embellishment_level': 'Medium',
 'embellishment': [],
 'embellishment_detailed': ['Gold zari woven polka dot pattern',
  'Intricate zari border work',
  'Metallic thread embroidery on pallu',
  'Subtle shimmering effect throughout'],
 'style': [],
 'brand_title': 'Red Dream',
 'title': 'Crim

In [5]:
import asyncio
import logging
import os
from pathlib import Path
import yaml
import pandas as pd
from typing import Dict, Any, Tuple, List, Optional
from dotenv import load_dotenv
import argparse
from datetime import datetime

from src.factories.clothing_factory import ClothingFactory
from src.services.api_service import APIService
from src.services.attribute_generator import AttributeGenerator
from src.services.batch_processor import BatchProcessor
from src.utils.logging_utils import (
    setup_logging, 
    LoggingContext, 
    PerformanceMetrics,
    StructuredLogger
)
from src.services.product_standardizer import ProductStandardizer
from src.base.clothing_item import ClothingItem


In [6]:

async def process_product(
    product_data: Dict[str, Any],
    product_type: str,
    attribute_generator: AttributeGenerator
) -> Optional[Dict[str, Any]]:
    """Process a single product."""
    product_id = product_data.get("product_id", "unknown")
    
    # Create product-specific metrics
    product_metrics = PerformanceMetrics()
    product_metrics.checkpoint("start")
    
    # Create structured logger for this product
    product_logger = StructuredLogger(
        "product_processor",
        extra_fields={
            "product_id": product_id,
            "product_type": product_type
        }
    )
    
    product_logger.info(f"Processing product {product_id}")
    
    # Generate attributes using the API
    try:
        product_metrics.checkpoint("api_call_start")
        generated_attributes = await attribute_generator.generate_attributes(
            product_data=product_data,
            product_type=product_type
        )
        product_metrics.checkpoint("api_call_end")
        
        # Calculate API call duration
        api_duration = product_metrics.measure(
            "api_call", 
            "api_call_start", 
            "api_call_end"
        )
        
        if generated_attributes:
            product_logger.info(
                f"Successfully generated attributes for product {product_id}",
                extra={
                    "duration": api_duration,
                    "attribute_count": len(generated_attributes)
                }
            )
            return generated_attributes
        else:
            product_logger.error(
                f"Failed to generate attributes for product {product_id}",
                extra={"duration": api_duration}
            )
            return None
    except Exception as e:
        product_logger.error(
            f"Error processing product {product_id}: {str(e)}",
            extra={"error_type": type(e).__name__},
            exc_info=True
        )
        return None
    finally:
        # Log total processing time
        total_duration = product_metrics.measure("total")
        product_logger.info(
            "Product processing completed",
            extra={"total_duration": total_duration}
        )



In [1]:
from src.utils.config_loader import ConfigManager


product_config = ConfigManager.get_product_config('saree')
defined_attributes = product_config.get('attributes', {})
defined_attributes

{'brand': {'required': True,
  'data_type': 'string',
  'item_type': 'string',
  'default': 'suta',
  'in_search_context': True},
 'brand_title': {'required': True,
  'data_type': 'string',
  'item_type': 'string',
  'in_search_context': True},
 'title': {'required': True,
  'data_type': 'string',
  'item_type': 'string',
  'in_search_context': True},
 'description': {'required': True,
  'data_type': 'string',
  'item_type': 'string',
  'in_search_context': True},
 'price': {'required': True,
  'data_type': 'float',
  'item_type': 'float',
  'min': 0,
  'in_search_context': True},
 'primary_color': {'required': True,
  'data_type': 'string',
  'item_type': 'enum',
  'allowed_values': 'Color',
  'threshold': 0.8,
  'in_search_context': True},
 'primary_color_hex': {'required': True,
  'data_type': 'string',
  'item_type': 'string',
  'pattern': '^#([A-Fa-f0-9]{6})$',
  'in_search_context': False},
 'primary_color_detailed': {'required': True,
  'data_type': 'string',
  'item_type': 'enu

In [4]:
ConfigManager.get_product_config('saree')['attributes']['brand']['default']

{'required': True,
 'data_type': 'string',
 'item_type': 'string',
 'default': 'suta',
 'in_search_context': True}

In [5]:
from src.base.clothing_item import ClothingItem
ClothingItem.build_search_context(product_data)

NameError: name 'product_data' is not defined

In [3]:
def test_prompt_loading():
    from pathlib import Path
    from src.services.attribute_generator import AttributeGenerator
    
    # Expanded config with required Cohere settings
    config = {
        "prompts": {
            "dir": "D:/OneDrive - StarHub Ltd/5. Scripts/product-attributes/config/prompts"  # Your actual path
        },
        "api": {
            "cohere": {
                "model": "embed-english-v3.0",  # or whatever model you're using
                "supported_operations": ["embed"]  # Add embed operation
            }
        }
    }
    
    # Create instance with minimal dependencies
    generator = AttributeGenerator(api_service=None, config=config)
    
    # Test prompt loading
    prompt = generator._load_prompt('saree')
    print(prompt)
    return prompt

# Run the test
prompt = test_prompt_loading()

You are a fashion expert specializing in Indian ethnic wear, particularly sarees. 
Analyze this saree image and product information to generate detailed JSON data.

IMPORTANT GUIDELINES:
- Focus ONLY on the saree in the image, not any blouse or other garments shown
- Use the exact product name from the data for brand_title (e.g. if product name is "Green Dream", use that)
- Analyze colors ONLY from the saree itself, not from any accompanying blouse or accessories
- Be precise in identifying embellishments and patterns that are actually present on the saree
- STRICTLY use only the allowed values provided for each attribute
- Use material information from the product data, don't try to infer it from the image
- For ruffled designs, use "Others" as pattern and "Sequinned" as embellishment
- Color pairings MUST be 6-digit hex codes (e.g., "#FF0000", "#00FF00")
- Use the exact product URL from the data for product_url field

Requirements:
1. Colors & Patterns:
   - "primary_color": Identify